In [2]:
import pandas as pd 

df = pd.read_csv(r"C:\Users\Aayush\Desktop\Support-ticket-copilot\data\customer_support_tickets.csv")
print(df.shape)
df.head()


(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
print("Missing values:")                 # just prints a heading
print(df.isnull().sum())                 # for each column, counts how many cells are empty

Missing values:
Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64


In [4]:
print("\nTicket types:")                 # heading (\n adds a blank line before it)
print(df["Ticket Type"].value_counts())  # counts how many tickets are in each type


Ticket types:
Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64


In [5]:
print("\nTicket priority:")
print(df["Ticket Priority"].value_counts())   # same, but for priority (Low / Medium / High / Critical)


Ticket priority:
Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64


In [6]:
print("\nSample descriptions:")
for text in df["Ticket Description"].sample(5, random_state=1):
    print("-", text)
    print()


Sample descriptions:
- I'm having an issue with the {product_purchased}. Please assist. Thanks in return."

"Thanks for understanding."

"Thank you for sharing."

"Sorry."

"Please do not forget. Sorry I've checked the device settings and made sure that everything is configured correctly.

- I'm having an issue with the {product_purchased}. Please assist. */ protected <StringProperty Name = "product_purchased_ID" Value = "5C0F4C7E2-845F- I've performed a factory reset on my {product_purchased}, hoping it would resolve the problem, but it didn't help.

- I'm encountering a software bug in the {product_purchased}. Whenever I try to perform a specific action, the application crashes. Are there any updates or fixes available?

It is possible to set up a specific update queue, such I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?

- I'm having an issue with the {product_purchased}. Please assist.

Pl

In [7]:
placeholders = df["Ticket Description"].str.findall(r"\{(\w+)\}").explode().value_counts()
print(placeholders)

Ticket Description
product_purchased           13873
error_message                 472
product_id                     52
product_name                   34
Product_purchased              18
                            ...  
Purchased                       1
product_purchased_amount        1
product_purchased_0             1
product_purchased_2             1
product_paid_1                  1
Name: count, Length: 191, dtype: int64


In [8]:
df["clean_description"] = df.apply(
    lambda row: row["Ticket Description"].replace("{product_purchased}", str(row["Product Purchased"])),
    axis=1,
)
df[["Ticket Description", "clean_description"]].head()

,Ticket Description,clean_description
0,I'm having an issue with the {product_purchase...,I'm having an issue with the GoPro Hero. Pleas...
1,I'm having an issue with the {product_purchase...,I'm having an issue with the LG Smart TV. Plea...
2,I'm facing a problem with my {product_purchase...,I'm facing a problem with my Dell XPS. The Del...
3,I'm having an issue with the {product_purchase...,I'm having an issue with the Microsoft Office....
4,I'm having an issue with the {product_purchase...,I'm having an issue with the Autodesk AutoCAD....


In [9]:
print(df["clean_description"].str.len().describe())
print("Unique descriptions:", df["clean_description"].nunique(), "of", len(df))

count    8469.000000
mean      281.972370
std        43.995224
min       135.000000
25%       263.000000
50%       290.000000
75%       310.000000
max       402.000000
Name: clean_description, dtype: float64
Unique descriptions: 8398 of 8469


In [10]:
print(df.groupby("Ticket Type")["Ticket Subject"].nunique())
print(df[["Ticket Type", "Ticket Subject", "clean_description"]].sample(5, random_state=2))

Ticket Type
Billing inquiry         16
Cancellation request    16
Product inquiry         16
Refund request          16
Technical issue         16
Name: Ticket Subject, dtype: int64
          Ticket Type    Ticket Subject  \
5735  Technical issue     Display issue   
7373   Refund request     Product setup   
8290  Billing inquiry    Hardware issue   
4289  Technical issue     Product setup   
6981  Product inquiry  Delivery problem   

                                      clean_description  
5735  I'm having an issue with the LG Smart TV. Plea...  
7373  I've noticed a software bug in the Sony PlaySt...  
8290  I'm facing a problem with my Google Pixel. The...  
4289  I've forgotten my password for my Nintendo Swi...  
6981  There seems to be a hardware problem with my D...  


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df["clean_description"]
y = df["Ticket Type"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vec = TfidfVectorizer(max_features=5000)
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

pred = model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.19244391971664698


In [12]:
d = df.dropna(subset=["Ticket Priority"])
X_tr, X_te, y_tr, y_te = train_test_split(d["clean_description"], d["Ticket Priority"], test_size=0.2, random_state=42)
vec2 = TfidfVectorizer(max_features=5000)
m2 = LogisticRegression(max_iter=1000).fit(vec2.fit_transform(X_tr), y_tr)
print("Priority accuracy:", accuracy_score(y_te, m2.predict(vec2.transform(X_te))))

Priority accuracy: 0.23730814639905548
